In [12]:
import requests
import pandas as pd
API_KEY = 'm5GkXnwARzqdEwbw5DCzW9UdiNlFnpFt'

def fetch_option_chain(symbol):
    url = f"https://api.polygon.io/v3/snapshot/options/{symbol}?apiKey=m5GkXnwARzqdEwbw5DCzW9UdiNlFnpFt"
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()

    return data

    # options = []

    # for contract in data["results"]:
    #     options.append({
    #         "strike": contract["details"]["strike_price"],
    #         "type": contract["details"]["contract_type"],
    #         "price": (contract["last_quote"]["bid"] +
    #                   contract["last_quote"]["ask"]) / 2
    #     })

    # return pd.DataFrame(options)

nvda = fetch_option_chain('NVDA')
nvda

{'results': [{'day': {'change': 1.75,
    'change_percent': 2.059,
    'close': 86.75,
    'high': 86.75,
    'last_updated': 1771822800000000000,
    'low': 86.75,
    'open': 86.75,
    'previous_close': 85,
    'volume': 3,
    'vwap': 86.75},
   'details': {'contract_type': 'call',
    'exercise_style': 'american',
    'expiration_date': '2026-02-23',
    'shares_per_contract': 100,
    'strike_price': 105,
    'ticker': 'O:NVDA260223C00105000'},
   'greeks': {},
   'open_interest': 7,
   'underlying_asset': {'ticker': 'NVDA'}},
  {'day': {'change': 0,
    'change_percent': 0,
    'close': 79.65,
    'high': 79.78,
    'last_updated': 1771621200000000000,
    'low': 79.65,
    'open': 79.78,
    'previous_close': 79.65,
    'volume': 9,
    'vwap': 79.7222},
   'details': {'contract_type': 'call',
    'exercise_style': 'american',
    'expiration_date': '2026-02-23',
    'shares_per_contract': 100,
    'strike_price': 110,
    'ticker': 'O:NVDA260223C00110000'},
   'greeks': {},
  

In [43]:
from massive import RESTClient

client = RESTClient("m5GkXnwARzqdEwbw5DCzW9UdiNlFnpFt")

options_data = []
for o in client.list_snapshot_options_chain(
    "NVDA",
    params={
        "order": "asc",
        "limit": 10,
        "sort": "ticker",
    },
):
    options_data.append({
        "ticker": o.details.ticker,
        "contract_type": o.details.contract_type,
        "strike_price": o.details.strike_price,
        "expiration_date": o.details.expiration_date,
        "underlying_ticker": o.underlying_asset.ticker,
        "open_interest": o.open_interest,
        "close": o.day.close if o.day else None,
        "volume": o.day.volume if o.day else None,
        "implied_volatility": o.implied_volatility,
        "last_updated": o.day.last_updated
    })

options_chain = pd.DataFrame(options_data)
options_chain

,ticker,contract_type,strike_price,expiration_date,underlying_ticker,open_interest,close,volume,implied_volatility,last_updated
0,O:NVDA260227C00050000,call,50.0,2026-02-27,NVDA,22,143.38,2.0,6.192947,1.771909e+18
1,O:NVDA260227C00055000,call,55.0,2026-02-27,NVDA,7,134.25,1.0,7.844806,1.771621e+18
2,O:NVDA260227C00060000,call,60.0,2026-02-27,NVDA,47,129.34,1.0,7.325070,1.771909e+18
3,O:NVDA260227C00065000,call,65.0,2026-02-27,NVDA,5,125.32,5.0,6.850166,1.771880e+18
4,O:NVDA260227C00070000,call,70.0,2026-02-27,NVDA,25,120.22,2.0,4.687584,1.771909e+18
...,...,...,...,...,...,...,...,...,...,...
3441,O:NVDA281215P00340000,put,340.0,2028-12-15,NVDA,525,157.07,2.0,0.459821,1.771909e+18
3442,O:NVDA281215P00350000,put,350.0,2028-12-15,NVDA,278,165.42,1.0,0.457222,1.771909e+18
3443,O:NVDA281215P00360000,put,360.0,2028-12-15,NVDA,373,174.25,1.0,0.454652,1.771909e+18
3444,O:NVDA281215P00370000,put,370.0,2028-12-15,NVDA,321,183.29,2.0,0.453999,1.771909e+18


In [2]:
import requests
import json

API_KEY = "m5GkXnwARzqdEwbw5DCzW9UdiNlFnpFt"

url = f"https://api.polygon.io/v3/snapshot/options/NVDA?apiKey={API_KEY}"

data = requests.get(url).json()

# print first contract fully formatted
print(json.dumps(data["results"][0], indent=2))

{
  "day": {
    "change": 2.13,
    "change_percent": 1.508,
    "close": 143.38,
    "high": 143.38,
    "last_updated": 1771909200000000000,
    "low": 140.02,
    "open": 140.02,
    "previous_close": 141.25,
    "volume": 2,
    "vwap": 141.7
  },
  "details": {
    "contract_type": "call",
    "exercise_style": "american",
    "expiration_date": "2026-02-27",
    "shares_per_contract": 100,
    "strike_price": 50,
    "ticker": "O:NVDA260227C00050000"
  },
  "greeks": {
    "delta": 0.9991883586352655,
    "gamma": 2.9785969940632345e-05,
    "theta": -0.043509966070854555,
    "vega": 0.0007156710739120724
  },
  "implied_volatility": 5.029932523803005,
  "open_interest": 22,
  "underlying_asset": {
    "ticker": "NVDA"
  }
}


In [36]:
import sqlite3

conn = sqlite3.connect("../data/options.db")
cursor = conn.cursor()

print("Connected successfully.")

Connected successfully.


In [37]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()

print(tables)

[('contracts',), ('snapshots',)]


In [39]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM snapshots", conn)

df

,snapshot_time,contract_symbol,underlying_symbol,day_change,day_change_percent,day_close,day_high,day_low,day_open,day_previous_close,day_volume,day_vwap,day_last_updated_ns,day_last_updated_utc,delta,gamma,theta,vega,implied_volatility,open_interest
0,2026-02-24T22:23:14.549527+00:00,O:NVDA260227C00050000,NVDA,2.13,1.508,143.38,143.38,140.02,140.02,141.25,2,141.700,1771909200000000000,2026-02-24T05:00:00+00:00,0.996293,0.000100,-0.200187,0.001899,6.190728,22
1,2026-02-24T22:23:14.549527+00:00,O:NVDA260227C00055000,NVDA,0.00,0.000,134.25,134.25,134.25,134.25,134.25,1,134.250,1771621200000000000,2026-02-20T21:00:00+00:00,0.982533,0.000308,-0.978347,0.009068,7.840966,7
2,2026-02-24T22:23:14.549527+00:00,O:NVDA260227C00060000,NVDA,-1.39,-1.060,129.34,129.34,129.34,129.34,130.73,1,129.340,1771909200000000000,2026-02-24T05:00:00+00:00,0.981167,0.000352,-0.973765,0.009552,7.321486,47
3,2026-02-24T22:23:14.549527+00:00,O:NVDA260227C00065000,NVDA,0.00,0.000,125.32,126.54,125.32,126.27,125.32,5,126.006,1771880400000000000,2026-02-23T21:00:00+00:00,0.979754,0.000401,-0.968148,0.010015,6.846817,5
4,2026-02-24T22:23:14.549527+00:00,O:NVDA260227C00070000,NVDA,-0.27,-0.224,120.22,120.22,120.15,120.15,120.49,2,120.185,1771909200000000000,2026-02-24T05:00:00+00:00,0.995199,0.000166,-0.193205,0.002286,4.685947,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3271,2026-02-24T22:23:14.549527+00:00,O:NVDA281215P00340000,NVDA,-4.50,-2.790,157.07,157.07,156.87,156.87,161.57,2,156.970,1771909200000000000,2026-02-24T05:00:00+00:00,-0.698940,0.003836,-0.012367,0.916444,0.459663,525
3272,2026-02-24T22:23:14.549527+00:00,O:NVDA281215P00350000,NVDA,-2.83,-1.680,165.42,165.42,165.42,165.42,168.25,1,165.420,1771909200000000000,2026-02-24T05:00:00+00:00,-0.722619,0.003918,-0.011452,0.887802,0.457055,278
3273,2026-02-24T22:23:14.549527+00:00,O:NVDA281215P00360000,NVDA,-3.68,-2.070,174.25,174.25,174.25,174.25,177.93,1,174.250,1771909200000000000,2026-02-24T05:00:00+00:00,-0.747129,0.004032,-0.010838,0.873691,0.454481,373
3274,2026-02-24T22:23:14.549527+00:00,O:NVDA281215P00370000,NVDA,-3.61,-1.930,183.29,183.29,183.07,183.07,186.90,2,183.180,1771909200000000000,2026-02-24T05:00:00+00:00,-0.768611,0.004107,-0.010205,0.871659,0.453826,321
